# Resume whatever is unfinished

Works for the matched-scale run, the seed sweep, or both, on whichever account
holds the quota. It looks at every checkpoint you attach, works out how far
each run got, and continues the ones that are incomplete.

Use this instead of notebook 3 when you are not certain what finished.

## Attach the inputs

**Input → Add Input**, and add:

1. Datasets → `aicd-code`
2. Every checkpoint source you have. Either the original notebook's output
   (Your Work → Notebooks) if it is on this account, or the checkpoint
   dataset you uploaded (see *Moving a checkpoint between accounts* in the
   README) if it is not.

Settings: `GPU T4 x2`, Internet `On`, then **Save Version → Save & Run All**.

## The 12-hour cap

`--max-hours` now stops training cleanly after the last epoch that fits,
instead of being killed partway through the next one. The first matched-scale
run lost 2.1 hours of GPU to a partial epoch that was thrown away; this stops
that happening again. Budget below is set to 9.5 h of training, leaving room
for evaluation inside a 12 h session.

In [ ]:
import os, sys, time, subprocess, shutil, pathlib, json

T0 = time.time()
def elapsed(label=""):
    m = (time.time() - T0) / 60
    print(f"[{m:6.1f} min] {label}", flush=True)

def run(cmd):
    """Run a pipeline stage and stop the notebook if it fails.

    Without the raise a failed stage prints a traceback and the next cell
    happily trains on whatever stale data is lying around.
    """
    print(">>", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, "-u", *[str(c) for c in cmd]])
    if r.returncode != 0:
        raise SystemExit(f"FAILED: {' '.join(str(c) for c in cmd)}")

print("Python", sys.version.split()[0])
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 in the right panel.")

## 1. Code

In [ ]:
WORK = pathlib.Path("/kaggle/working/project")
WORK.mkdir(parents=True, exist_ok=True)

def find_aicd():
    root = pathlib.Path("/kaggle/input")
    if not root.exists():
        return None
    for cand in root.rglob("aicd"):
        if (cand / "config.py").exists() and (cand / "models").is_dir():
            return cand
    return None

src = find_aicd()
if src is None:
    raise SystemExit(
        "aicd/ not found under /kaggle/input.\n"
        "Add your code dataset: right panel -> Input -> Add Input -> Datasets,\n"
        "then search for the dataset you created from aicd-code.zip.")

dest = WORK / "aicd"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(src, dest)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code ->", dest)
print("configs present:", sorted(p.name for p in (dest/"configs").glob("kaggle*.yaml")))

## 2. Dependencies

In [ ]:
pkgs = ["xgboost", "tree-sitter", "tree-sitter-language-pack",
        "datasets", "shap", "pyyaml", "scikit-learn", "pyarrow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import importlib
for m in ["xgboost", "sklearn", "transformers", "datasets", "yaml"]:
    mod = importlib.import_module(m)
    print(f"  ok  {m:14s} {getattr(mod, '__version__', '')}")
elapsed("deps")

## 3. Take stock

Finds every checkpoint and every corpus among the attached inputs, and reports
what each run still needs. Nothing trains until you have seen this table.

In [ ]:
art = WORK / "aicd" / "artifacts"
(art / "data").mkdir(parents=True, exist_ok=True)
IN = pathlib.Path("/kaggle/input")

# A run is identified by its tag, which is embedded in the checkpoint name.
# Keep the newest copy if the same tag appears in more than one input.
found = {}
for p in IN.rglob("branch_a_*_ckpt.pt"):
    tag = p.name[len("branch_a_"):-len("_ckpt.pt")]
    if tag not in found or p.stat().st_mtime > found[tag].stat().st_mtime:
        found[tag] = p

# Each tag needs the corpus it was trained on. matched used three shards,
# the seeds used one, and the two are different files with the same name.
corpora = sorted(IN.rglob("splits.parquet"), key=lambda p: p.stat().st_size,
                 reverse=True)

print(f"checkpoints found: {len(found)}")
print(f"corpora found    : {len(corpora)}")
for c in corpora:
    print(f"    {c.stat().st_size/1e6:8.0f} MB  {c}")
print()

if not found:
    raise SystemExit(
        "No branch_a_*_ckpt.pt in any attached input.\n"
        "Attach the notebook output, or the checkpoint dataset, that holds it.")

TOTAL_EPOCHS = 3
plan = []
print(f"{'tag':12s} {'epochs done':>12s} {'remaining':>10s}  {'est. hours':>10s}")
print("-" * 50)
for tag, p in sorted(found.items()):
    ck = torch.load(p, map_location="cpu", weights_only=False)
    done = ck["epoch"] + 1
    del ck
    left = TOTAL_EPOCHS - done
    # 4.9 h/epoch for matched (394,624 rows), 2.4 h/epoch for a seed (196,854).
    per = 4.9 if tag == "matched" else 2.4
    print(f"{tag:12s} {done:>7d} of {TOTAL_EPOCHS} {left:>10d}  {left*per:>10.1f}")
    if left > 0 or True:      # even a finished run still needs evaluation
        plan.append((tag, p, left, per))
print()
elapsed("stock taken")

## 4. Choose what to run this session

`matched` first: it is the experiment that decides the paper's tier. Edit
`ONLY` below if you want to force a particular run.

In [ ]:
ONLY = None          # e.g. "matched" or "seed1"; None = everything found

order = {"matched": 0}
plan.sort(key=lambda t: order.get(t[0], 1))
todo = [t for t in plan if ONLY is None or t[0] == ONLY]

BUDGET_H = 9.5       # training hours; leaves ~2 h for evaluation under the cap

# Ask the trainer what it supports rather than assuming. An attached code
# dataset can be an older version than the one on your laptop, and passing a
# flag it has never heard of kills the run in the first two minutes.
_h = subprocess.run([sys.executable, "-m", "aicd.models.modernbert_triplet",
                     "--help"], capture_output=True, text=True)
HAS_MAX_HOURS = "--max-hours" in (_h.stdout + _h.stderr)
HAS_TAG = "--tag" in (_h.stdout + _h.stderr)

if not HAS_TAG:
    raise SystemExit(
        "The attached aicd-code dataset is too old: it has no --tag flag.\n"
        "Upload kaggle/aicd-code.zip as a New Version of that dataset, then\n"
        "in this notebook's Input panel switch it to the latest version.")
if not HAS_MAX_HOURS:
    print("NOTE: this code dataset predates --max-hours. Training will run")
    print("      without a clean stop, so a long run may be killed mid-epoch")
    print("      and lose that epoch's progress. Upload the current zip to")
    print("      fix. Continuing anyway; nothing is at risk.\n")

print("this session will run:")
for tag, _, left, per in todo:
    print(f"  {tag:12s} {left} epoch(s) left, about {left*per:.1f} h")
print(f"\ntraining budget: {BUDGET_H} h  (12 h session cap)")
if sum(l * p for _, _, l, p in todo) > BUDGET_H:
    print("\nThis exceeds the budget. --max-hours will stop cleanly after the")
    print("last epoch that fits, and the rest resumes next session.")

## 5. Restore and run

For each run: copy in its checkpoint and the matching corpus, then train with
`--resume`. The corpus is chosen by size — the matched build is roughly twice
the single-shard one — so the wrong data cannot be paired with a checkpoint.

In [ ]:
for tag, ck_src, left, per in todo:
    print("\n" + "=" * 60)
    print(f"  {tag}")
    print("=" * 60, flush=True)

    shutil.copy(ck_src, art / f"branch_a_{tag}_ckpt.pt")
    print(f"checkpoint <- {ck_src.name}  ({ck_src.stat().st_size/1e9:.2f} GB)")

    # Largest corpus for matched, smallest for a seed run.
    src = corpora[0] if tag == "matched" else corpora[-1]
    shutil.copy(src, art / "data" / "splits.parquet")
    import pandas as pd
    n = len(pd.read_parquet(art / "data" / "splits.parquet", columns=["split"]))
    print(f"corpus     <- {src}  ({n:,} rows total)")

    cfg = "kaggle_matched.yaml" if tag == "matched" else f"kaggle_{tag}.yaml"
    cmd = ["-m", "aicd.models.modernbert_triplet", "--config", cfg,
           "--tag", tag, "--resume"]
    if HAS_MAX_HOURS:
        cmd += ["--max-hours", str(BUDGET_H)]
    run(cmd)
    elapsed(f"{tag} done for this session")

## 6. Where each run stands now

In [ ]:
rep = WORK / "aicd" / "eval" / "reports"
ORIG = {"s1_in_distribution": 0.8977, "s2_unseen_generator": 0.8685,
        "s3_unseen_language": 0.5667, "s4_unseen_domain": 0.4029,
        "s5_compound": 0.2378}

done_any = False
for f in sorted(rep.glob("branch_a_*.json")):
    tag = f.stem[len("branch_a_"):]
    r = json.load(open(f))["slices"]
    if "s5_compound" not in r:
        continue
    done_any = True
    s1, s5 = r["s1_in_distribution"]["macro_f1"], r["s5_compound"]["macro_f1"]
    print(f"\n{tag}:  S1 {s1:.4f} -> S5 {s5:.4f}   drop {s1-s5:.4f}")
    if tag == "matched":
        print(f"  original (196,854 rows): 0.8977 -> 0.2378   drop 0.6599")
        print(f"  matched  (394,624 rows), a factor of 2.0")
        if s5 < 0.45:
            print("  COLLAPSE PERSISTS. Training-set size is excluded.")
        else:
            print("  COLLAPSE DOES NOT PERSIST. Report it; the framing changes.")

if not done_any:
    print("No run finished all 3 epochs yet. Checkpoints are saved.")
    print("Re-run this notebook next session and it continues from here.")

## 7. Save

In [ ]:
OUT = pathlib.Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

reports = WORK / "aicd" / "eval" / "reports"
if reports.exists():
    shutil.copytree(reports, OUT / "reports", dirs_exist_ok=True)

# The probability arrays are what the analysis modules re-read at home, and
# they are small. The model weights are hundreds of MB and are not needed to
# reproduce any number in the paper, so they stay behind.
art = WORK / "aicd" / "artifacts"
npy = OUT / "arrays"
npy.mkdir(exist_ok=True)
n = 0
for f in art.glob("proba_a*.npy"):
    shutil.copy(f, npy / f.name); n += 1
for f in art.glob("labels.parquet"):
    shutil.copy(f, npy / f.name)
if (art / "kaggle").exists():
    for f in (art / "kaggle").glob("*"):
        if f.is_file() and f.stat().st_size < 200e6:
            shutil.copy(f, npy / f.name); n += 1

shutil.make_archive("/kaggle/working/results", "zip", OUT)
print(f"copied {n} arrays")
print("-> /kaggle/working/results.zip  (download this from the Output tab)")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.stat().st_size/1024:8.0f} KB  {p.relative_to(OUT)}")
elapsed("saved")